# CSE 438 · Part B — Final Comparison (label-efficiency)

**Group 01 · Dept. of CSE, East West University**

Merges every SSL method's `results.json` with the Part-A supervised baseline and answers the central question: *how close does each SSL method (fine-tuned on only the val split) get to full supervision, using far fewer labels?* Attach: each method notebook's output + the Part-A `results.json`.

In [ ]:
# ===== Part B final comparison: label-efficiency vs Part-A supervised baseline =====
import json, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})
PARTA_BASELINE_MIOU = 0.768; PARTA_N_TRAIN = 13447   # DeepLabV3 supervised (fallback if not found in results)
CLASS_NAMES = ["background", "liver", "tumor"]

def extract_miou(e):
    if not isinstance(e, dict): return None
    if isinstance(e.get("best_ckpt"), dict) and "mIoU" in e["best_ckpt"]: return e["best_ckpt"]["mIoU"]
    for k in ("epoch50_test_mIoU", "mIoU", "test_mIoU", "miou"):
        if isinstance(e.get(k), (int, float)): return e[k]
    return None
def extract_pc(e):
    if isinstance(e.get("best_ckpt"), dict) and "per_class_iou" in e["best_ckpt"]: return e["best_ckpt"]["per_class_iou"]
    return e.get("per_class_iou")

merged = {}
for f in glob.glob("/kaggle/input/**/results.json", recursive=True):
    try: merged.update(json.load(open(f)))
    except Exception as ex: print("skip", f, ex)
print("methods found:", list(merged.keys()))
# ---- version pins (PDF §4 item 1: "setup & imports with version pins") ----
import torch as _t, numpy as _n, pandas as _p
try: import albumentations as _a; _av = _a.__version__
except Exception: _av = "n/a"
try: import transformers as _tr; _tv = _tr.__version__
except Exception: _tv = "n/a"
print(f"torch {_t.__version__} | numpy {_n.__version__} | pandas {_p.__version__} "
      f"| albumentations {_av} | transformers {_tv}")

In [ ]:
# ===== VIZ — validated palette + figure style =====
# Palette validated with the dataviz validator (light, surface #ffffff, --pairs all):
#   4 series  #2a78d6/#eb6834/#1baf7a/#4a3aa7 -> CVD dE 9.2, normal dE 16.3  ALL PASS
#   mask      liver #eda100 vs tumor #e34948  -> CVD dE 15.3, normal dE 20.8 ALL PASS
import matplotlib as mpl
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
try:
    from IPython.display import display
except Exception:
    display = print

C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_YELLOW, C_RED = "#eda100", "#e34948"
INK, INK2, MUTED, GRIDC, RULE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
# colour follows the ENTITY, never its rank — a filtered chart never repaints survivors
METHOD_COLORS = {"SimCLR": C_BLUE, "BYOL": C_ORANGE, "MAE": C_AQUA, "DINOv2": C_VIOLET}
CLASS_COLORS  = ["#000000", C_YELLOW, C_RED]          # background / liver / tumour
cmap = ListedColormap(CLASS_COLORS)
CLASS_LEGEND = [Patch(facecolor=C_YELLOW, label="liver"), Patch(facecolor=C_RED, label="tumour")]
# error map: identity carried by legend + label, never colour alone (CVD warn band)
ERR_COLORS = {"correct": "#141413", "liver_err": C_YELLOW, "tumor_fn": C_RED, "tumor_fp": C_AQUA}
ERR_LEGEND = [Patch(facecolor=C_RED,    label="tumour missed (FN)"),
              Patch(facecolor=C_AQUA,   label="tumour false-positive (FP)"),
              Patch(facecolor=C_YELLOW, label="liver error")]

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "font.family": "sans-serif",
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlepad": 10,
    "axes.labelcolor": INK2, "axes.labelsize": 9.5, "axes.labelpad": 6,
    "axes.edgecolor": RULE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRIDC, "grid.linewidth": 0.7, "grid.alpha": 1.0,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 2.0, "lines.markersize": 5,
})

def _fmt_df(df, fmt=None):
    """Per-column formatting: ints get thousands separators, floats 4dp unless overridden."""
    fmt = fmt or {}
    d = df.copy()
    for c in d.columns:
        k = d[c].dtype.kind
        f = fmt.get(c) if isinstance(fmt, dict) else None
        if f is None and k in "iu": f = "{:,.0f}"
        if f is None and k == "f":  f = "{:,.0f}" if d[c].dropna().gt(1000).all() else "{:.4f}"
        if f: d[c] = d[c].map(lambda v, f=f: "—" if pd.isna(v) else f.format(v))
    return d.astype(str)

def styled_table(df, title="", best=None, fmt=None):
    """Publication-style HTML table (Kaggle renders pandas Styler natively)."""
    fmt = fmt or {}
    spec = {}
    for c in df.columns:
        k = df[c].dtype.kind
        if c in fmt: spec[c] = fmt[c]
        elif k in "iu": spec[c] = "{:,.0f}"
        elif k == "f": spec[c] = "{:.4f}"
    txt = [c for c in df.columns if df[c].dtype.kind not in "iuf"]
    s = (df.style.format(spec, na_rep="—").hide(axis="index").set_caption(title)
         .set_properties(subset=txt, **{"text-align": "left"})
         .set_table_styles([
            {"selector": "caption", "props": [("caption-side","top"),("font-weight","700"),
                ("font-size","12.5pt"),("padding","4px 0 10px"),("color",INK),("text-align","left")]},
            {"selector": "th", "props": [("background","#f0efec"),("color",INK),("font-weight","600"),
                ("text-align","right"),("padding","7px 12px"),("border","none"),
                ("border-bottom",f"1.5px solid {RULE}"),("white-space","nowrap")]},
            {"selector": "td", "props": [("text-align","right"),("padding","7px 12px"),("border","none"),
                ("font-variant-numeric","tabular-nums"),("color",INK),("white-space","nowrap")]},
            {"selector": "tbody tr:nth-child(even)", "props": [("background","#fafaf8")]},
            {"selector": "", "props": [("border-collapse","collapse"),("font-size","10.5pt")]},
         ]))
    for c in ([best] if isinstance(best, str) else (best or [])):
        if c in df.columns and df[c].dtype.kind == "f":
            s = s.highlight_max(subset=[c], props="background:#dff0e4;font-weight:700;")
    return s

def table_png(df, path, title="", best=None, fmt=None):
    """Report-ready PNG of the same table. Column widths follow content so nothing clips."""
    d = _fmt_df(df, fmt)
    widths = np.array([max(len(str(c)), *(len(v) for v in d[c])) for c in d.columns], float)
    widths = widths / widths.sum()
    fig_w = min(15.0, 1.05 * sum(max(len(str(c)), *(len(v) for v in d[c])) for c in d.columns) * 0.115 + 1.2)
    fig, ax = plt.subplots(figsize=(max(7.0, fig_w), 0.62 + 0.40 * len(d)))
    ax.axis("off"); ax.set_xmargin(0); ax.set_ymargin(0)
    t = ax.table(cellText=d.values, colLabels=list(d.columns), loc="center",
                 cellLoc="right", colWidths=list(widths))
    t.auto_set_font_size(False); t.set_fontsize(9.5); t.scale(1, 1.5)
    txt_idx = [i for i, c in enumerate(df.columns) if df[c].dtype.kind not in "iuf"]
    bi = {}
    for c in ([best] if isinstance(best, str) else (best or [])):
        if c in df.columns and df[c].dtype.kind == "f":
            bi[list(df.columns).index(c)] = int(df[c].idxmax())
    for (r, c), cell in t.get_celld().items():
        cell.set_linewidth(0)
        if c in txt_idx: cell._loc = "left"
        if r == 0:
            cell.set_facecolor("#f0efec"); cell.set_text_props(weight="bold", color=INK)
            cell.visible_edges = "B"; cell.set_edgecolor(RULE); cell.set_linewidth(1.4)
        else:
            cell.set_facecolor("#fafaf8" if r % 2 == 0 else "white")
            if c in bi and r - 1 == bi[c]:
                cell.set_facecolor("#dff0e4"); cell.set_text_props(weight="bold")
    if title: ax.set_title(title, fontweight="bold", fontsize=12, loc="left", pad=10)
    plt.savefig(path); plt.show()
print("VIZ ready — validated palette, publication table styles, figure defaults.")

## 1. Label-efficiency table (Task E)

In [ ]:
# ===== label-efficiency table =====
SSL = ["simclr", "byol", "mae", "dinov2"]
partA = None
for k in ("deeplabv3", "deeplab", "DeepLabV3"):
    if k in merged and extract_miou(merged[k]): partA = extract_miou(merged[k]); break
partA = partA or PARTA_BASELINE_MIOU
rows = [{"model": "Part-A DeepLabV3 (supervised)", "regime": "full supervision", "n_labelled": PARTA_N_TRAIN,
         "test_mIoU": round(partA, 4), "tumor_IoU": None, "%of_baseline": 100.0}]
for k in SSL:
    if k not in merged: continue
    e = merged[k]; mi = extract_miou(e); pc = extract_pc(e)
    rows.append({"model": e.get("method", k), "regime": "SSL + fine-tune",
                 "n_labelled": e.get("n_labelled_finetune"), "test_mIoU": round(mi, 4) if mi else None,
                 "tumor_IoU": round(pc[2], 4) if pc else None,
                 "%of_baseline": round(100*mi/partA, 1) if mi else None})
tbl = pd.DataFrame(rows); tbl.to_csv("partB_label_efficiency.csv", index=False)
DISPLAY = {"model":"Model","regime":"Regime","n_labelled":"Labelled imgs",
           "test_mIoU":"Test mIoU","tumor_IoU":"Tumour IoU","%of_baseline":"% of baseline"}
tbl_disp = tbl.rename(columns=DISPLAY)
display(styled_table(tbl_disp, "Label-efficiency: SSL (val-only labels) vs Part-A full supervision",
                     best=["Test mIoU", "Tumour IoU"], fmt={"% of baseline": "{:.1f}"}))
ratio = (rows[1]["n_labelled"]/PARTA_N_TRAIN) if len(rows) > 1 and rows[1]["n_labelled"] else None
if ratio: print(f"\nlabel-efficiency: SSL methods fine-tune on {rows[1]['n_labelled']} labelled "
                 f"= {100*ratio:.0f}% of the {PARTA_N_TRAIN} used by full supervision.")

# rendered table figure
table_png(tbl_disp, "fig_label_efficiency_table.png",
          title="Part B — label-efficiency vs Part-A full supervision",
          best=["Test mIoU", "Tumour IoU"], fmt={"% of baseline": "{:.1f}"})

## 2. Charts + best-method verdict

In [ ]:
# ===== charts: mIoU vs baseline + per-class IoU =====
sslrows = [r for r in rows if r["regime"] == "SSL + fine-tune" and r["test_mIoU"] is not None]
if sslrows:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    names = [r["model"] for r in sslrows]; mious = [r["test_mIoU"] for r in sslrows]
    cols = [METHOD_COLORS.get(n, C_BLUE) for n in names]
    ax[0].bar(names, mious, color=cols, width=.62, zorder=3)
    ax[0].axhline(partA, color=INK2, ls=(0, (5, 3)), lw=1.6, zorder=2)
    ax[0].text(len(names)-0.45, partA, f"  Part-A supervised {partA:.3f}", va="bottom", ha="right",
               fontsize=8.5, color=INK2)
    for i, v in enumerate(mious): ax[0].text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    ax[0].set_ylabel("test mIoU"); ax[0].set_title("SSL (val-only labels) vs full supervision"); ax[0].legend()
    tum = [r["tumor_IoU"] for r in sslrows]
    ax[1].bar(names, tum, color=cols, width=.62, zorder=3); ax[1].set_ylabel("tumour IoU")
    ax[1].set_title("tumor IoU by SSL method")
    plt.tight_layout(); plt.savefig("fig_label_efficiency_bars.png", dpi=160); plt.show()

    # per-class IoU grouped bars where available
    pcs = {r["model"]: extract_pc(merged.get(k, {})) for k, r in zip([x for x in SSL if x in merged], sslrows)}
    pcs = {m: v for m, v in pcs.items() if v}
    if pcs:
        x = np.arange(3); w = 0.8/max(len(pcs),1)
        plt.figure(figsize=(8, 4.5))
        for i, (m, v) in enumerate(pcs.items()):
            plt.bar(x + i*w, v, w*0.9, label=m, color=METHOD_COLORS.get(m, C_BLUE), zorder=3)
        plt.xticks(x + w*(len(pcs)-1)/2, CLASS_NAMES); plt.ylabel("IoU"); plt.title("Per-class IoU across SSL methods")
        plt.legend(fontsize=8); plt.tight_layout(); plt.savefig("fig_per_class_iou.png", dpi=160); plt.show()

    best = max(sslrows, key=lambda r: r["test_mIoU"])
    print(f"\nBest SSL method: {best['model']} — test mIoU {best['test_mIoU']:.4f} "
          f"({best['%of_baseline']:.1f}% of full-supervision) using ~{100*ratio:.0f}% of the labels.")
    json.dump({"baseline_mIoU": partA, "table": rows}, open("partB_comparison.json", "w"), indent=2)
else:
    print("No SSL results found yet — run the method notebooks and attach their outputs.")

## 3. Task F — consolidated error analysis (REQUIRED)

Assignment §4: the final notebook must produce the **consolidated error analysis**. We intersect each method's
worst-30 test slices to test whether the four SSL methods fail on the **same** images (data-driven difficulty) or
**different** ones (architecture/pretext-specific), and compare that against the Part-A supervised failure mode.

In [ ]:
# ===== Task F (consolidated): do the SSL methods fail on the same slices? =====
import itertools
worst_sets, per_image = {}, {}
for k in SSL:
    e = merged.get(k, {})
    ea = e.get("error_analysis") if isinstance(e, dict) else None
    if ea and ea.get("worst30_slice_ids"): worst_sets[e.get("method", k)] = set(ea["worst30_slice_ids"])
for f in glob.glob("/kaggle/input/**/*_per_image_iou.csv", recursive=True):
    per_image[Path(f).name.split("_per_image")[0]] = pd.read_csv(f)

if len(worst_sets) >= 2:
    names = list(worst_sets)
    J = np.eye(len(names))
    for i, j in itertools.combinations(range(len(names)), 2):
        a, b = worst_sets[names[i]], worst_sets[names[j]]
        J[i, j] = J[j, i] = len(a & b) / max(len(a | b), 1)
    shared = set.intersection(*worst_sets.values())
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(J, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{J[i,j]:.2f}", ha="center", color="white" if J[i,j] > .5 else "black", fontsize=8)
    plt.colorbar(im); ax.set_title("Task F — worst-30 slice overlap (Jaccard)")
    plt.tight_layout(); plt.savefig("fig_taskF_failure_overlap.png", dpi=160); plt.show()
    print(f"slices in EVERY method's worst-30: {len(shared)}")
    print("High Jaccard => difficulty is data-driven (tiny lesions); low => pretext-specific failure modes.")
else:
    print("Attach the method notebooks' outputs (results.json with error_analysis) for consolidated Task F.")

# per-image IoU agreement between methods (scatter, first pair available)
if len(per_image) >= 2:
    ks = list(per_image)[:2]
    a = per_image[ks[0]][["slice_id", "miou"]].rename(columns={"miou": ks[0]})
    b = per_image[ks[1]][["slice_id", "miou"]].rename(columns={"miou": ks[1]})
    mrg = a.merge(b, on="slice_id")
    r = np.corrcoef(mrg[ks[0]], mrg[ks[1]])[0, 1]
    plt.figure(figsize=(5.4, 5))
    plt.scatter(mrg[ks[0]], mrg[ks[1]], s=9, alpha=.35, color=C_BLUE, edgecolors="none")
    plt.plot([0,1],[0,1], ls=(0,(5,3)), color=MUTED, lw=1.4); plt.xlabel(f"{ks[0]} per-image mIoU")
    plt.ylabel(f"{ks[1]} per-image mIoU"); plt.title(f"failure agreement  (r = {r:.3f})")
    plt.tight_layout(); plt.savefig("fig_taskF_agreement_scatter.png", dpi=160); plt.show()
    print(f"Pearson r = {r:.3f} — high r means the methods find the SAME slices hard.")

## Summary
The table and bars quantify label-efficiency: SSL methods reach a large fraction of the Part-A supervised mIoU (0.768) while fine-tuning on only ~19% of the labels. Discuss which pretext task (contrastive / negative-free / masked / self-distillation) transfers best to liver+tumour segmentation in the report's Insights section.

---

# 🔬 EXTRA — additional research visualisations

> ### ⚠️ NOT REQUIRED BY THE ASSIGNMENT
> Everything **above** this line satisfies Tasks A–F of *Assignment Part B*. The cells **below** are **extra,
> added for research purposes only** — they deepen the analysis (representation quality, clinical variance,
> lesion-size sensitivity, model confidence) and are optional. They can be skipped without affecting compliance.

In [ ]:
# 🔬 EXTRA (research) — radar across metrics + label-efficiency positioning
if sslrows:
    # radar
    axes_m = ["mIoU", "mean Dice", "pixel acc", "liver IoU", "tumor IoU"]
    def vec(k):
        e = merged.get(k, {}); bc = e.get("best_ckpt", {}) if isinstance(e, dict) else {}
        pc = bc.get("per_class_iou") or [np.nan]*3
        return [bc.get("mIoU", np.nan), bc.get("mean_dice", np.nan), bc.get("pixel_acc", np.nan), pc[1], pc[2]]
    have = [k for k in SSL if k in merged and merged[k].get("best_ckpt")]
    if have:
        ang = np.linspace(0, 2*np.pi, len(axes_m), endpoint=False).tolist(); ang += ang[:1]
        fig = plt.figure(figsize=(6.4, 6.4)); ax = plt.subplot(polar=True)
        for k in have:
            v = vec(k); v = [0 if (x is None or (isinstance(x, float) and np.isnan(x))) else x for x in v]
            ax.plot(ang, v + v[:1], marker="o", label=merged[k].get("method", k)); ax.fill(ang, v + v[:1], alpha=.08)
        ax.set_xticks(ang[:-1]); ax.set_xticklabels(axes_m, fontsize=8); ax.set_ylim(0, 1)
        ax.set_title("EXTRA · multi-metric profile", pad=18); ax.legend(loc="lower right", fontsize=7)
        plt.tight_layout(); plt.savefig("fig_EXTRA_radar.png", dpi=160); plt.show()

    # label-efficiency positioning: mIoU vs number of labels (log x)
    plt.figure(figsize=(7, 4.6))
    plt.scatter([PARTA_N_TRAIN], [partA], s=110, marker="*", color=INK, zorder=4,
                label=f"Part-A supervised ({partA:.3f})")
    for r_ in sslrows:
        plt.scatter([r_["n_labelled"]], [r_["test_mIoU"]], s=60, zorder=3)
        plt.annotate(r_["model"], (r_["n_labelled"], r_["test_mIoU"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=8)
    plt.xscale("log"); plt.xlabel("labelled images used (log)"); plt.ylabel("test mIoU")
    plt.title("EXTRA · label-efficiency frontier"); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig("fig_EXTRA_label_efficiency_frontier.png", dpi=160); plt.show()
    print("Points far LEFT at similar height = strong label-efficiency (near-baseline mIoU with far fewer labels).")